# 01 — Reference arms on the identical frozen proxy

* `baseline`: organisers' starter (the `exp_ryu/` working tree).
* `ryu_final`: ryu's committed final pipeline (`origin/ryu` bd90a9c, `015_final_pipeline/run.py` + `weights_new`: TCN ensemble trained by ryu on Metrica + SkillCorner) applied zero-shot with all of ryu's constants; λ is calibrated, as ryu does, to the training matches' possession rate.
* `rasya_c06`, `rasya_c08`: rasya's best deployable pipelines from `exp_rasya/.../012_spellf1` with the parameters rasya froze (touch radius 0.8 m, minlen 30, merge 150; dead cue out 15 frames or slow <2 m/s for 25 / 400 frames). No re-tuning here.
* `gbm_const`: exp_0's feature family `deadball` + LightGBM (frozen snapshot), decoded with the exp_1 constant-penalty Viterbi selected inside training folds.

In [1]:
from pathlib import Path
import os, sys, json, time
ROOT = Path.cwd()
if not (ROOT / 'exp1_common.py').exists():
    raise RuntimeError('Run with working directory exp_jeremy/exp_1')
sys.path.insert(0, str(ROOT))
sys.dont_write_bytecode = True
import numpy as np
import pandas as pd
pd.set_option('display.width', 180)
pd.set_option('display.max_columns', 30)
import exp1_common as C
import exp1_protocol as P

## Rule arms

In [2]:
rule_rows = pd.DataFrame(P.evaluate_rules())
rule_rows.to_csv(C.RESULTS / 'rule_rows.csv', index=False)
print(rule_rows.pivot(index='match', columns='variant', values='spell_f1').round(4).to_string())
print(rule_rows.groupby('variant')[['spell_f1', 'macro_f1', 'n_pred', 'n_true']].mean().round(4).to_string())

variant  baseline  rasya_c06  rasya_c08
match                                  
J03WMX     0.1805     0.3103     0.3223
J03WN1     0.1716     0.3013     0.3075
J03WOH     0.1898     0.3169     0.3179
J03WOY     0.1671     0.3163     0.3224
J03WPY     0.1860     0.3468     0.3541
J03WQQ     0.2148     0.3383     0.3202
J03WR9     0.1930     0.3259     0.3302
           spell_f1  macro_f1    n_pred    n_true
variant                                          
baseline     0.1861    0.4579  511.1429  437.2857
rasya_c06    0.3223    0.6342  278.2857  437.2857
rasya_c08    0.3249    0.5915  255.2857  437.2857


## ryu's shipped TCN pipeline, zero-shot

In [3]:
import warnings
t0 = time.time()
with warnings.catch_warnings():
    warnings.simplefilter('ignore', RuntimeWarning)
    ryu_rows = pd.DataFrame(P.evaluate_ryu())
ryu_rows.to_csv(C.RESULTS / 'ryu_rows.csv', index=False)
print(ryu_rows[['match', 'spell_f1', 'macro_f1', 'n_true', 'n_pred', 'ryu_lambda', 'ryu_target_possessions']].round(4).to_string(index=False))
print('mean spellF1', round(ryu_rows.spell_f1.mean(), 4), f'{time.time() - t0:.0f}s')

 match  spell_f1  macro_f1  n_true  n_pred  ryu_lambda  ryu_target_possessions
J03WMX    0.4489    0.8838     511     241      0.4649                289.4567
J03WN1    0.4043    0.8471     500     251      0.6204                280.7195
J03WOH    0.5144    0.8894     402     217      0.3827                272.0993
J03WOY    0.4490    0.8610     456     238      0.5032                282.6529
J03WPY    0.5088    0.8902     421     230      0.2500                289.9406
J03WQQ    0.5422    0.8867     401     243      0.4686                282.2742
J03WR9    0.5441    0.8927     370     227      0.2979                291.1284
mean spellF1 0.4874 28s


## exp_0 LightGBM arm (21 pair fits + 7 outer fits)

In [4]:
t0 = time.time()
gbm_predictions = P.ensure_predictions('gbm')
result = P.evaluate_variant('gbm_const', gbm_predictions)
C.write_json(C.RESULTS / 'variant_gbm_const.json', result)
print(pd.DataFrame(result['rows'])[['match', 'spell_f1', 'macro_f1', 'n_true', 'n_pred', 'lambda_team', 'lambda_dead']].round(4).to_string(index=False))
print(f'{time.time() - t0:.0f}s')

 match  spell_f1  macro_f1  n_true  n_pred  lambda_team  lambda_dead
J03WMX    0.4180    0.8729     511     290          1.0          8.0
J03WN1    0.4020    0.8280     500     311          1.0          8.0
J03WOH    0.4867    0.8791     402     267          1.0          8.0
J03WOY    0.4138    0.8497     456     299          1.0         16.0
J03WPY    0.4826    0.8748     421     257          1.0          4.0
J03WQQ    0.5053    0.8837     401     302          1.0          4.0
J03WR9    0.4953    0.8863     370     272          1.0          4.0
20s
